In [0]:
%run ../silver/00_silver_helpers

In [0]:
df= read_table('sales')
display(df)

In [0]:
df.printSchema()

In [0]:
print('Updating data-types of the columns')
print('Trimming string objects if any leading/trailing spaces \n')
print('............ \n')

df= df.select(
          trim(col('order_id').try_cast('string')).alias('order_id'),
          trim(col('customer_id').try_cast('string')).alias('customer_id'),
          trim(col('product_id').try_cast('string')).alias('product_id'),
          trim(col('store_id').try_cast('string')).alias('store_id'),
          col('order_date').try_cast('date').alias('order_date'),
          col('quantity').try_cast('integer').alias('quantity'),
          col('unit_price').try_cast('decimal(10,2)').alias('unit_price'),
          col('discount').try_cast('decimal(10,2)').alias('discount'),
          trim(col('payment_method').try_cast('string')).alias('payment_method'),
          col('_ingestion_timestamp').try_cast('timestamp').alias('_ingestion_timestamp'),
          col('_source_file').try_cast('string').alias('_source_file')
          )
print('Updated data-types of the columns')
print('Trimmed string objects if any leading/trailing spaces \n')

print(f'Duplicate orders count : {df.count() - df.dropDuplicates(["order_id"]).count()}')
df = df.dropDuplicates(["order_id"])
print('Duplicate orders dropped \n')

print('............ \n')

print('Removing records having quantity/unit_price 0/(-ve)')
df= df.where(col('quantity')> 0).where(col('unit_price')>=0)
print(f'Records dropped : {(df.where(col('quantity')> 0).where(col('unit_price')>=0)).count()} \n')

print('............ \n')


print(f'Checking for nulls in "discount" column: {df.where(isnull(col('discount'))).count()}')
if df.where(isnull(col('discount'))).count() >0 :
    print('Filling null discounts with 0')
    df= df.fillna(0, ['discount'])
    print('Null discounts filled with 0 \n')

else:
    print('No nulls in discount column \n')

print('............ \n')


print(f'Checking for nulls in "payment_method" column: {df.where(isnull(col('payment_method'))).count()}')
if df.where(isnull(col('payment_method'))).count() >0 :
    print('Filling null payment_methods with "UNKNOWN"')
    df= df.fillna("UNKNOWN", ['payment_method'])
    print('Null payment_methods filled with "UNKNOWN"')

else:
    print('No nulls in payment_method column \n')

In [0]:
df.printSchema()

In [0]:
display(df)

In [0]:
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}')
print(f'Created schema {catalog_name}.{schema_name} \n')

save_table(df,'sales_clean')